<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# AfriCare Support Analytics
## Notebook 2 — Data Cleaning & Feature Engineering
### ✅ VERSION CORRIGÉE

---

> **Comment lire ce notebook :**
> Chaque bloc de code est précédé d'une cellule markdown qui explique le **pourquoi**
> avant le **comment**. Les blocs `🎓 POINT PÉDAGOGIQUE` contiennent les règles généralisables
> à tout projet data. Lis toujours l'explication avant d'exécuter le code.

---

| Info | Détail |
|---|---|
| **Prérequis** | Notebook 1 complété — les 5 datasets chargés et audités |
| **Objectif** | Produire un dataset fiable, enrichi et prêt pour l'analyse SQL et le ML |
| **Sortie attendue** | `support_clean_analytics.csv` + table SQL `fact_support_analytics` |
| **Durée estimée** | 3h à 4h |
| **Compétences** | Nettoyage de données · Feature engineering · Logique métier · Export SQL |

---

### 🗺️ Plan du notebook

```
1. Imports & configuration
2. Chargement des données
3. Audit qualité (rapport synthétique)
4. Nettoyage
   4.1 Suppression des doublons
   4.2 Traitement des anomalies métier
5. Construction de la table analytique
   5.1 Fusion des tables
   5.2 Feature engineering temporel
   5.3 Variable cible ML (ticket_at_risk)
   5.4 Flags business
   5.5 Sélection des colonnes finales
6. Synthèse
```

---

# 1. Imports & configuration

> 🎓 **Pourquoi configurer l'environnement en premier ?**
>
> Regrouper tous les imports et la configuration en tête de notebook a deux avantages :
>
> **Lisibilité** — n'importe qui qui ouvre le notebook voit immédiatement les dépendances.
>
> **Reproductibilité** — si un collègue doit réexécuter le notebook sur une autre machine,
> il sait exactement quelles librairies installer.
>
> `warnings.filterwarnings("ignore")` : on supprime les avertissements pandas/numpy
> qui polluent l'output sans apporter d'information utile en production.
> En développement initial, on peut les laisser actifs pour détecter des usages dépréciés.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

import os
import sys

# Détecter si on est dans Colab ou en local
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = "/content/drive/MyDrive/DataProjectLab/projects/customer_support_analytics/"
else:
    SAVE_PATH = "./outputs/"

# Chemin pour enregistrer les fichiers exportés
os.makedirs(SAVE_PATH, exist_ok=True)
print(f" Environnement Colab : {'Colab' if IN_COLAB else 'Local'}")
print(f" Dossier de travail : {SAVE_PATH}")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

print("✅ Environnement prêt")

 Environnement Colab : Local
 Dossier de travail : ./outputs/
✅ Environnement prêt


---

# 2. Chargement des données

> 🎓 **On recharge les données depuis les CSV bruts, pas depuis le Notebook 1.**
>
> Chaque notebook doit être **autonome** — exécutable indépendamment sans dépendre
> de variables définies dans un autre notebook. C'est une règle de bonne pratique
> fondamentale en data engineering.
>
> Si on partait des variables du Notebook 1, le Notebook 2 serait inutilisable
> si on l'ouvre seul. Et en production, les notebooks sont souvent exécutés
> automatiquement par des orchestrateurs (Airflow, Prefect) sans ordre garanti.
>
> **Rappel `parse_dates` :** obligatoire sur toutes les colonnes de dates.
> Sans ce paramètre, `created_at` est chargé comme `object` (string).
> Tout calcul temporel ultérieur échouerait silencieusement ou produirait des résultats faux.

In [2]:
BASE_URL = "https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/main/projets/customer_support_analytics/data/"

agents       = pd.read_csv(BASE_URL + "agents.csv")
categories   = pd.read_csv(BASE_URL + "categories.csv")
tickets      = pd.read_csv(BASE_URL + "tickets.csv",      parse_dates=['created_at'])
interactions = pd.read_csv(BASE_URL + "interactions.csv", parse_dates=['timestamp'])
sla_alerts   = pd.read_csv(BASE_URL + "sla_alerts.csv",  parse_dates=['alert_timestamp'])

print(f"{'Table':<15} {'Lignes':>8} {'Colonnes':>10}")
print("-" * 36)
for name, df in [('agents', agents), ('categories', categories), ('tickets', tickets),
                  ('interactions', interactions), ('sla_alerts', sla_alerts)]:
    print(f"{name:<15} {len(df):>8,} {len(df.columns):>10}")

Table             Lignes   Colonnes
------------------------------------
agents                12          7
categories            10          7
tickets           15,287         19
interactions      15,779          6
sla_alerts         2,000          5


---

# 3. Audit qualité

> 🎓 **Pourquoi faire un audit ici si on l'a déjà fait dans le Notebook 1 ?**
>
> Dans le Notebook 1, l'audit était **exploratoire** : on découvrait les données,
> on identifiait les problèmes potentiels, on documentait les actions nécessaires.
>
> Dans le Notebook 2, l'audit est **opérationnel** : on vérifie que les données
> chargées sont conformes à ce qu'on attend avant de lancer le nettoyage.
> C'est un filet de sécurité — si quelqu'un a modifié les CSV depuis le Notebook 1,
> on le détecte immédiatement.
>
> **Bonne pratique :** toujours commencer un notebook de traitement par une vérification
> rapide des dimensions et de la qualité. Ça ne prend que quelques secondes et peut
> éviter des heures de débogage.

### Rapport synthétique de qualité

> 🎓 **La fonction `quality_report` ci-dessous construit un tableau de synthèse.**
>
> Plutôt que d'afficher les mêmes informations pour chaque table séparément
> (ce qui prend de la place et est difficile à lire), on les consolide en un seul
> tableau avec les métriques clés : lignes, colonnes, valeurs manquantes, doublons.
>
> Ce tableau est ce qu'on partage avec son équipe pour dire :
> **"Voici l'état des données avant nettoyage."**

In [3]:
quality_report = []
for name, df in [('agents', agents), ('categories', categories), ('tickets', tickets),
                  ('interactions', interactions), ('sla_alerts', sla_alerts)]:
    quality_report.append({
        "dataset":          name,
        "lignes":           df.shape[0],
        "colonnes":         df.shape[1],
        "valeurs_nulles":   int(df.isnull().sum().sum()),
        "pct_nulles":       round(df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2),
        "doublons_ligne":   int(df.duplicated().sum()),
    })

rapport = pd.DataFrame(quality_report)
print(rapport.to_string(index=False))

     dataset  lignes  colonnes  valeurs_nulles  pct_nulles  doublons_ligne
      agents      12         7               0        0.00               0
  categories      10         7               0        0.00               0
     tickets   15287        19            6742        2.32               0
interactions   15779         6               0        0.00               0
  sla_alerts    2000         5               0        0.00               0


### Zoom sur les valeurs manquantes de `tickets`

> 🎓 **Toutes les valeurs manquantes ne se traitent pas de la même façon.**
>
> Il faut distinguer trois catégories de nulls :
>
> **Null légitime** — la valeur n'est pas applicable dans ce contexte.
> Exemple : `csat` est null pour les tickets non résolus — il est **normal** qu'un ticket
> en cours n'ait pas de note de satisfaction. On ne doit **pas** imputer ces valeurs.
>
> **Null technique** — la valeur devrait être présente mais manque à cause d'une erreur
> de collecte ou d'export. Exemple : `first_response_heures` null sur un ticket résolu.
> On doit imputer ou signaler ces valeurs.
>
> **Null critique** — la valeur manque sur une colonne qui sera utilisée comme variable
> cible ou feature ML. Exemple : `sla_breach` null. Ce cas doit être traité en priorité
> car il corromprait l'entraînement du modèle.

In [4]:
print("=== Valeurs manquantes — TICKETS ===")
missing = tickets.isnull().sum()
missing_pct = (missing / len(tickets) * 100).round(2)
rapport_nulls = pd.DataFrame({
    'nulls': missing,
    'pct': missing_pct,
    'criticité': missing.apply(lambda v: '🔴 Critique' if v > 0 and missing.index[missing == v][0] in
                 ['sla_breach','ratio_sla','sla_heures'] else
                 ('🟡 Légitime' if missing.index[missing == v][0] == 'csat' else
                 ('🟠 À traiter' if v > 0 else '✅ OK')))
}).query("nulls > 0").sort_values("pct", ascending=False)

print(rapport_nulls.to_string())
print("Légende : 🔴 = bloquant ML | 🟠 = à imputer | 🟡 = normal (csat)")

=== Valeurs manquantes — TICKETS ===
      nulls   pct   criticité
csat   6742 44.10  🟡 Légitime
Légende : 🔴 = bloquant ML | 🟠 = à imputer | 🟡 = normal (csat)


### Contrôles métier

> 🎓 **Les contrôles métier vérifient la cohérence logique entre colonnes.**
>
> On vérifie trois règles que les données doivent respecter par définition :
>
> **Règle 1 — Délais négatifs impossibles.**
> `resolution_heures < 0` est physiquement impossible. Un ticket ne peut pas être résolu
> avant d'être créé. Ces valeurs indiquent un bug dans le calcul ou un problème de fuseau horaire.
>
> **Règle 2 — Cohérence `sla_breach` / `ratio_sla`.**
> Par construction, `sla_breach = 1` si et seulement si `ratio_sla > 1`.
> Si ces deux colonnes sont incohérentes, nos KPIs de breach seront faux — dans un sens ou l'autre.
>
> **Règle 3 — CSAT dans [1, 5].**
> La note CSAT ne peut valoir que 1, 2, 3, 4 ou 5. Une valeur hors de cette plage
> est une erreur de saisie ou de conversion.

In [5]:
print("=== Contrôles métier ===\n")

# Règle 1 : délais négatifs
delais_negatifs = tickets[tickets["resolution_heures"] < 0]
print(f"  Délais de résolution négatifs : {len(delais_negatifs):>4}",
      "✅" if len(delais_negatifs) == 0 else "⚠️ → remplacement par médiane en NB2")

# Règle 2 : cohérence sla_breach / ratio_sla (les deux sens)
incoh_non_flag = tickets[(tickets["ratio_sla"] > 1)  & (tickets["sla_breach"] == 0)].shape[0]
incoh_sur_flag = tickets[(tickets["ratio_sla"] <= 1) & (tickets["sla_breach"] == 1)].shape[0]
print(f"  ratio_sla>1 mais sla_breach=0 : {incoh_non_flag:>4}",
      "✅" if incoh_non_flag == 0 else "⚠️ → breach sous-comptabilisés")
print(f"  ratio_sla≤1 mais sla_breach=1 : {incoh_sur_flag:>4}",
      "✅" if incoh_sur_flag == 0 else "⚠️ → faux positifs breach")

# Règle 3 : CSAT hors [1-5]
csat_invalide = tickets[tickets["csat"].notna() & ~tickets["csat"].between(1, 5)].shape[0]
print(f"  CSAT hors [1-5]               : {csat_invalide:>4}",
      "✅" if csat_invalide == 0 else "⚠️ → lignes à exclure")

# Bonus : distribution des statuts (vérification des valeurs)
print(f"\n  Valeurs uniques de statut :")
for v, n in tickets["statut"].value_counts().items():
    print(f"    '{v}' : {n:,}")

=== Contrôles métier ===

  Délais de résolution négatifs :    0 ✅
  ratio_sla>1 mais sla_breach=0 :    0 ✅
  ratio_sla≤1 mais sla_breach=1 :    2 ⚠️ → faux positifs breach
  CSAT hors [1-5]               :    0 ✅

  Valeurs uniques de statut :
    'Résolu' : 8,068
    'Escaladé' : 3,323
    'En cours' : 2,293
    'Backlog' : 1,126
    'Fermé' : 477


On a 2 tickets incohérents sur la cohérence SLA.
C'est marginal — 2 sur 15 287. On les note et on les traite.

Et la distribution des statuts confirme ce qu'on avait vu :
8 068 Résolus, 3 323 Escaladés, 2 293 En cours, 1 126 en Backlog, 477 Fermés.

Le chiffre qui attire l'œil immédiatement : **3 323 tickets escaladés**.
C'est 21% du total. Ce sera un point central de notre analyse dans le Notebook 3.

---

# 4. Nettoyage

> 🎓 **La règle d'or du nettoyage : toujours justifier chaque décision.**
>
> Nettoyer sans justification est dangereux pour deux raisons :
>
> **Raison 1 — Auditabilité.** Si quelqu'un relit votre code dans 6 mois (ou si c'est vous
> dans 6 mois), vous devez comprendre pourquoi vous avez fait chaque choix.
>
> **Raison 2 — Reproductibilité.** Une décision de nettoyage non documentée sera
> peut-être faite différemment par un collègue sur les mêmes données. Le dataset final
> ne sera pas le même.
>
> Pour chaque opération de nettoyage, ce notebook suit le format :
> → **Quoi** : quelle anomalie on traite
> → **Pourquoi** : quel problème ça poserait si on ne le traitait pas
> → **Comment** : quelle stratégie de nettoyage on choisit et pourquoi cette stratégie

## 4.1 — Suppression des doublons

> 🎓 **Deux types de doublons, deux stratégies différentes.**
>
> **Doublons de clé primaire** (`duplicated(subset=['ticket_id'])`) :
> Deux tickets avec le même `ticket_id` — l'un des deux est un fantôme.
> Stratégie : on garde la première occurrence (`keep='first'`) et on supprime les suivantes.
>
> **Doublons de ligne complète** (`duplicated()` sans argument) :
> Deux lignes identiques sur toutes les colonnes — souvent un artefact d'export ou de concaténation.
> Stratégie : suppression directe, aucune information n'est perdue.
>
> **Pourquoi `drop_duplicates(subset='ticket_id')` et non `drop_duplicates()` ?**
> Parce que deux tickets peuvent avoir le même ID mais des valeurs différentes sur d'autres colonnes
> (par exemple si le statut a été mis à jour entre deux exports). Dans ce cas, `drop_duplicates()`
> garderait les deux lignes alors qu'on veut en garder une seule par ticket.
> `subset='ticket_id'` cible exactement le problème.

In [6]:
# Comptage avant nettoyage
print("AVANT nettoyage des doublons :")
print(f"  tickets      : {len(tickets):>6,} lignes")
print(f"  agents       : {len(agents):>6,} lignes")
print(f"  categories   : {len(categories):>6,} lignes")
print(f"  interactions : {len(interactions):>6,} lignes")
print(f"  sla_alerts   : {len(sla_alerts):>6,} lignes")

# Suppression ciblée sur clé primaire
tickets      = tickets.drop_duplicates(subset="ticket_id",      keep="first").reset_index(drop=True)
agents       = agents.drop_duplicates(subset="agent_id",        keep="first").reset_index(drop=True)
categories   = categories.drop_duplicates(subset="category_id", keep="first").reset_index(drop=True)
interactions = interactions.drop_duplicates().reset_index(drop=True)
sla_alerts   = sla_alerts.drop_duplicates(subset="ticket_id",   keep="first").reset_index(drop=True)

print("\nAPRÈS nettoyage des doublons :")
print(f"  tickets      : {len(tickets):>6,} lignes")
print(f"  agents       : {len(agents):>6,} lignes")
print(f"  categories   : {len(categories):>6,} lignes")
print(f"  interactions : {len(interactions):>6,} lignes")
print(f"  sla_alerts   : {len(sla_alerts):>6,} lignes")

AVANT nettoyage des doublons :
  tickets      : 15,287 lignes
  agents       :     12 lignes
  categories   :     10 lignes
  interactions : 15,779 lignes
  sla_alerts   :  2,000 lignes

APRÈS nettoyage des doublons :
  tickets      : 15,287 lignes
  agents       :     12 lignes
  categories   :     10 lignes
  interactions : 15,779 lignes
  sla_alerts   :  2,000 lignes


## 4.2 — Traitement des anomalies métier

> 🎓 **Pourquoi imputer par la médiane et pas par la moyenne ?**
>
> Les délais de résolution (`resolution_heures`) ont probablement une distribution
> **asymétrique à droite** — la majorité des tickets se résolvent rapidement, mais
> quelques tickets très complexes créent une longue queue de distribution.
>
> Dans une distribution asymétrique :
> - La **moyenne** est tirée vers le haut par les valeurs extrêmes → elle surestimerait
>   le délai "typique" si on l'utilisait pour imputer.
> - La **médiane** est robuste aux outliers → elle représente mieux le comportement central.
>
> **Règle générale :** pour imputer des valeurs numériques avec outliers, préférer la médiane.
> Pour des distributions symétriques (ce qui est rare en pratique), la moyenne est acceptable.
>
> **Pourquoi `.mask()` plutôt qu'un filtre + assignation ?**
> `df["col"].mask(condition, np.nan)` remplace par NaN là où la condition est vraie.
> C'est plus lisible et plus sûr qu'un filtre booléen avec assignation directe
> (qui peut déclencher un `SettingWithCopyWarning` sur certaines versions de pandas).

In [7]:
# Imputation des délais négatifs par la médiane
mediane_resolution = tickets["resolution_heures"].median()
print(f"Médiane resolution_heures : {mediane_resolution:.1f} heures")

# .mask(condition) remplace par NaN là où la condition est vraie
# .fillna(médiane) remplace ensuite les NaN par la médiane
n_avant = (tickets["resolution_heures"] < 0).sum()
tickets["resolution_heures"] = (tickets["resolution_heures"]
                                  .mask(tickets["resolution_heures"] < 0, np.nan)
                                  .fillna(mediane_resolution))
n_apres = (tickets["resolution_heures"] < 0).sum()

print(f"Délais négatifs avant : {n_avant} | après : {n_apres} ✅")

# Normalisation du champ statut (espaces parasites fréquents en CRM)
# .str.strip() supprime les espaces en début et fin de chaîne
# .str.title() ou .str.strip() seul — ici strip suffit
tickets["statut"] = tickets["statut"].astype(str).str.strip()

print(f"\nValeurs de statut après normalisation : {sorted(tickets['statut'].unique())}")

Médiane resolution_heures : 167.0 heures
Délais négatifs avant : 0 | après : 0 ✅

Valeurs de statut après normalisation : ['Backlog', 'En cours', 'Escaladé', 'Fermé', 'Résolu']


---

# 5. Construction de la table analytique

> 🎓 **Qu'est-ce qu'une "table analytique" et pourquoi en construire une ?**
>
> Une table analytique (ou "fact table enrichie") est une table dénormalisée qui
> contient en une seule ligne **toutes les informations nécessaires** pour analyser
> un enregistrement — ici, un ticket.
>
> Avantages :
> - **Simplicité des requêtes** : plus besoin de joins dans SQL ou Power BI pour
>   chaque analyse — tout est déjà dans une seule table.
> - **Performance** : moins de jointures = requêtes plus rapides sur de grands volumes.
> - **Cohérence** : toutes les analyses partent du même dataset nettoyé.
>
> **La différence avec le modèle en étoile :**
> Le modèle en étoile (dim + fact séparées) est optimal pour les entrepôts de données
> en production. La table analytique dénormalisée est optimale pour l'exploration,
> l'analyse ad-hoc et le machine learning — c'est l'usage qu'on en fait ici.

## 5.1 — Fusion des tables

> 🎓 **`how='left'` dans le merge — un choix toujours délibéré.**
>
> On part de `tickets` (table principale) et on enrichit avec les informations
> d'`agents` et de `categories`. On utilise `how='left'` pour **conserver tous les tickets**,
> même ceux dont l'`agent_id` ou le `category_id` serait absent de la table de référence.
>
> **Alternative `how='inner'` :** ne garderait que les tickets avec un agent ET une catégorie
> référencés. Si l'audit a détecté des orphelins (FK sans correspondance), un inner join
> les supprimerait silencieusement — réduisant le volume sans avertissement.
>
> Avec un left join, les orphelins restent dans le dataset avec des NaN sur les colonnes
> de l'agent ou de la catégorie — ce qui est **visible et traçable**.
>
> **`suffixes=("", "_cat")`** : quand deux tables fusionnées ont des colonnes de même nom
> (ici `nom` existe dans `agents` et `categories`), pandas ajoute automatiquement un suffixe.
> On configure ce suffixe explicitement pour avoir `nom` (agent) et `nom_cat` (catégorie)
> plutôt que les `nom_x` et `nom_y` par défaut — bien moins lisibles.

In [8]:
# Sélection des colonnes utiles avant merge (évite les colonnes redondantes)
agents_cols     = ["agent_id", "nom", "tier", "bureau", "csat_moyen",
                    "taux_resolution_pct", "tickets_actifs"]
categories_cols = ["category_id", "nom", "domaine", "priorite_defaut", "sla_strict", "sla_heures"]

df = (tickets
      .merge(agents[agents_cols],          on="agent_id",    how="left")
      .merge(categories[categories_cols],  on="category_id", how="left",
             suffixes=("", "_cat")))

print(f"Shape après fusion : {df.shape}")
print(f"Tickets conservés  : {len(df):,} / {len(tickets):,}")

# Vérification que le left join n'a pas créé de doublons
assert len(df) == len(tickets), "⚠️ Le merge a créé des doublons — vérifier les FK"
print("✅ Aucun doublon introduit par le merge")

Shape après fusion : (15287, 30)
Tickets conservés  : 15,287 / 15,287
✅ Aucun doublon introduit par le merge


## 5.2 — Feature Engineering temporel

> 🎓 **Pourquoi créer des features temporelles ?**
>
> La colonne `created_at` contient un horodatage complet — elle encode beaucoup
> d'informations implicites que le modèle ML ne peut pas exploiter directement
> (un timestamp brut est un entier de grande valeur, sans signification structurelle).
>
> En décomposant le timestamp en features, on rend ces patterns **explicites** :
>
> **`heure_creation`** — les tickets créés à 2h du matin ont un comportement différent
> de ceux créés à 10h. La couverture des agents est différente, le contexte d'urgence aussi.
>
> **`jour_semaine`** (0=lundi, 6=dimanche) — les tickets du lundi matin cumulent
> les problèmes du weekend non traités. Le vendredi soir, les agents sont moins nombreux.
>
> **`est_weekend`** (0/1) — flag binaire directement exploitable par le ML.
> Un modèle d'arbre de décision peut séparer simplement weekend vs semaine.
>
> **`heure_hors_bureau`** — le créneau 8h-18h est la plage nominale de support.
> Hors de ce créneau, les SLA sont plus difficiles à tenir (moins d'agents disponibles).
>
> 🎓 **Le `.astype(int)` sur un booléen** est une bonne pratique pour le ML.
> Les modèles scikit-learn préfèrent les entiers 0/1 aux booléens `True/False`
> pour les features binaires — ça évite des avertissements et assure la compatibilité
> avec toutes les librairies ML.

In [9]:
# Features temporelles — décomposition du timestamp
df["heure_creation"]    = df["created_at"].dt.hour
df["jour_semaine"]      = df["created_at"].dt.dayofweek       # 0=Lundi, 6=Dimanche
df["mois"]              = df["created_at"].dt.month
df["est_weekend"]       = (df["jour_semaine"] >= 5).astype(int)
df["heure_hors_bureau"] = ((df["heure_creation"] < 8) |
                             (df["heure_creation"] > 18)).astype(int)

# Encodage ordinal du tier (pour ML)
# Tier 1 < Tier 2 < Tier 3 : relation ordinale logique (niveau d'expertise)
df["tier_num"] = df["tier"].map({"Tier 1": 1, "Tier 2": 2, "Tier 3": 3})

print("Features temporelles créées :")
for col in ["heure_creation", "jour_semaine", "mois", "est_weekend",
             "heure_hors_bureau", "tier_num"]:
    non_null = df[col].notna().sum()
    print(f"  {col:<25} : {non_null:,} valeurs non-null")

# Distribution de l'heure de création (vérification)
print(f"\nHeure min : {df['heure_creation'].min()} | max : {df['heure_creation'].max()}")
print(f"Tickets le weekend : {df['est_weekend'].sum():,} ({df['est_weekend'].mean()*100:.1f}%)")
print(f"Tickets hors bureau : {df['heure_hors_bureau'].sum():,} ({df['heure_hors_bureau'].mean()*100:.1f}%)")

Features temporelles créées :
  heure_creation            : 15,287 valeurs non-null
  jour_semaine              : 15,287 valeurs non-null
  mois                      : 15,287 valeurs non-null
  est_weekend               : 15,287 valeurs non-null
  heure_hors_bureau         : 15,287 valeurs non-null
  tier_num                  : 15,287 valeurs non-null

Heure min : 0 | max : 23
Tickets le weekend : 2,945 (19.3%)
Tickets hors bureau : 3,768 (24.6%)


## 5.3 — Variable cible ML : `ticket_at_risk`

> 🎓 **La variable cible est la décision la plus importante du projet ML.**
>
> Elle définit ce que le modèle apprend à prédire. Une variable cible mal définie
> produit un modèle techniquement correct mais inutile en production.
>
> Voici la définition retenue et sa justification :
>
> ```python
> ticket_at_risk = sla_breach | in_backlog | (statut == 'Escaladé')
> ```
>
> **`sla_breach = 1`** → le ticket a dépassé son délai SLA.
> C'est le signal le plus direct de problème — une violation contractuelle ou de qualité de service.
>
> **`in_backlog = 1`** → le ticket est bloqué dans la file d'attente.
> Un ticket en backlog est à risque même s'il n'a pas encore breché son SLA — il le fera
> probablement si personne n'intervient.
>
> **`statut == 'Escaladé'`** → le ticket a dû être escaladé à un niveau supérieur.
> L'escalade indique une complexité ou une urgence que l'agent initial n'a pas pu gérer.
> C'est un signal que la résolution sera plus longue et que le client est probablement frustré.
>
> 🎓 **Pourquoi l'union (`|`) et pas l'intersection (`&`) ?**
> On veut détecter **tous les tickets problématiques**, quelle que soit la forme du problème.
> L'intersection ne garderait que les tickets qui vérifient les trois critères simultanément —
> ce serait beaucoup trop restrictif.
>
> 🎓 **Important : définir la variable cible AVANT le feature engineering.**
> Si on créait des features après la variable cible, on risquerait de créer des features
> qui "fuitent" de l'information cible dans les inputs du modèle (data leakage).
> L'ordre correct est toujours : nettoyage → variable cible → features → ML.

In [10]:
# Création de la variable cible
df["ticket_at_risk"] = (
    (df["sla_breach"] == 1) |          # SLA dépassé
    (df["in_backlog"] == 1) |          # Ticket bloqué en backlog
    (df["statut"] == "Escaladé")       # Ticket nécessitant escalade
).astype(int)

# Distribution de la variable cible
n_risk    = df["ticket_at_risk"].sum()
n_total   = len(df)
pct_risk  = n_risk / n_total * 100

print(f"Distribution de ticket_at_risk :")
print(f"  Tickets à risque     : {n_risk:,}  ({pct_risk:.1f}%)")
print(f"  Tickets non à risque : {n_total - n_risk:,}  ({100 - pct_risk:.1f}%)")
print(f"  Total                : {n_total:,}")

# Détail de la composition du risque
print(f"\nComposition du risque :")
print(f"  Via sla_breach=1          : {(df['sla_breach'] == 1).sum():,}")
print(f"  Via in_backlog=1          : {(df['in_backlog'] == 1).sum():,}")
print(f"  Via statut=Escaladé       : {(df['statut'] == 'Escaladé').sum():,}")

# Vérification équilibre des classes (important pour le ML)
ratio = min(pct_risk, 100-pct_risk) / max(pct_risk, 100-pct_risk)
balance = "✅ Équilibré" if ratio > 0.3 else "⚠️ Déséquilibré — envisager SMOTE ou class_weight"
print(f"\nÉquilibre des classes (ratio) : {ratio:.2f} → {balance}")

Distribution de ticket_at_risk :
  Tickets à risque     : 8,250  (54.0%)
  Tickets non à risque : 7,037  (46.0%)
  Total                : 15,287

Composition du risque :
  Via sla_breach=1          : 7,230
  Via in_backlog=1          : 1,126
  Via statut=Escaladé       : 3,323

Équilibre des classes (ratio) : 0.85 → ✅ Équilibré


La moitié des tickets sont à risque.

C'est un dataset **quasi-équilibré** — ce qui est idéal pour l'entraînement d'un modèle ML.

Pas besoin de techniques de rééchantillonnage comme SMOTE.

Un ticket est à risque si au moins une de ces trois conditions est vraie :
il a breaché son SLA, il est en backlog, ou il a été escaladé.

C'est une variable composite — elle capture différentes **formes d'échec** du service support.

## 5.4 — Flags business

> 🎓 **Un flag business, c'est une règle métier encodée en 0/1.**
>
> Ces variables ne sont pas des features ML au sens strict — elles sont créées
> pour l'analyse SQL et le dashboard Power BI, pour permettre des filtres rapides
> et des KPIs précis.
>
> Elles serviront aussi de features ML secondaires, mais leur vraie valeur est
> dans la **lisibilité** : un superviseur comprend immédiatement "is_high_priority"
> sans avoir à savoir que c'est `priorite <= 2`.
>
> **`is_high_priority`** — les tickets de priorité 1 ou 2. Ces tickets ont des SLA
> plus stricts et un impact client plus fort en cas de breach.
>
> **`is_long_wait`** — seuil à 4 heures pour la première réponse. Au-delà, l'expérience
> client se dégrade significativement (benchmark secteur : SLA first response ≤ 4h
> pour 80% des tickets en support B2C).
>
> **`is_reopened`** — un ticket rouvert est un signal fort d'insatisfaction.
> La première résolution n'a pas convaincu le client — il a dû revenir.
>
> 🎓 **`.fillna(0)` sur `reopened` :** si `reopened` est null, on considère que le ticket
> n'a pas été rouvert (null = absence d'information = pas de réouverture documentée).
> C'est une **imputation par valeur par défaut** cohérente avec la sémantique du champ.

In [11]:
# Flags business
df["is_high_priority"] = (df["priorite"] <= 2).astype(int)
df["is_long_wait"]     = (df["first_response_heures"] > 4).astype(int)
df["is_reopened"]      = df["reopened"].fillna(0).astype(int)

print("Flags business créés et distribution :")
for flag, label in [("is_high_priority", "Priorité haute (1-2)"),
                      ("is_long_wait",     "Attente > 4h"),
                      ("is_reopened",      "Ticket rouvert")]:
    n   = df[flag].sum()
    pct = df[flag].mean() * 100
    print(f"  {flag:<22} : {n:>5,} tickets ({pct:>5.1f}%) — {label}")

Flags business créés et distribution :
  is_high_priority       : 6,485 tickets ( 42.4%) — Priorité haute (1-2)
  is_long_wait           : 14,246 tickets ( 93.2%) — Attente > 4h
  is_reopened            : 1,181 tickets (  7.7%) — Ticket rouvert


## 5.5 — Sélection des colonnes finales

> 🎓 **Pourquoi sélectionner les colonnes explicitement ?**
>
> Le dataset fusionné contient de nombreuses colonnes. Certaines sont redondantes
> (ex : `sla_heures` et `sla_heures_cat` sont la même information après le merge).
> D'autres sont inutiles pour l'analyse ou le ML (ex : clés internes de jointure).
>
> Sélectionner explicitement les colonnes finales a trois avantages :
> **Lisibilité** — on sait exactement ce que contient le dataset final.
> **Taille** — un CSV plus petit est plus rapide à charger et partager.
> **Clarté des intentions** — les colonnes incluses signalent ce qui est pertinent.
>
> **La convention de nommage dans le dataset final :**
> - Colonnes sources : noms originaux des CSV (`sla_breach`, `ratio_sla`)
> - Variables créées : préfixe ou suffixe explicite (`ticket_at_risk`, `is_*`, `est_*`)
> - Colonnes enrichies de `agents` : sans suffixe (ex : `tier`, `csat_moyen`)
> - Colonnes enrichies de `categories` : suffixe `_cat` si collision de nom (ex : `nom_cat`)

In [12]:
# Colonnes finales — regroupées par thème pour la lisibilité
cols_ticket    = ["ticket_id", "created_at", "category_id", "agent_id", "pays", "canal",
                   "priorite", "statut", "sla_heures", "first_response_heures",
                   "resolution_heures", "sla_breach", "nb_contacts", "reopened",
                   "csat", "in_backlog", "ratio_sla"]

cols_agent     = ["nom", "tier", "tier_num", "bureau", "csat_moyen",
                   "taux_resolution_pct", "tickets_actifs"]

cols_categorie = ["nom_cat", "domaine", "priorite_defaut", "sla_strict"]

cols_features  = ["heure_creation", "jour_semaine", "mois", "est_weekend", "heure_hors_bureau"]

cols_cibles    = ["ticket_at_risk", "is_high_priority", "is_long_wait", "is_reopened"]

# Assemblage avec vérification que chaque colonne existe bien
all_cols = cols_ticket + cols_agent + cols_categorie + cols_features + cols_cibles
cols_manquantes = [c for c in all_cols if c not in df.columns]
if cols_manquantes:
    print(f"⚠️  Colonnes absentes du DataFrame : {cols_manquantes}")
    all_cols = [c for c in all_cols if c in df.columns]

final_df = df[all_cols].copy()

print(f"Shape finale : {final_df.shape}")
print(f"  {len(cols_ticket)} colonnes tickets sources")
print(f"  {len(cols_agent)} colonnes agents")
print(f"  {len(cols_categorie)} colonnes catégories")
print(f"  {len(cols_features)} features temporelles")
print(f"  {len(cols_cibles)} variables cibles/flags")
print(f"\nAperçu des 3 premières lignes :")
final_df.head(5)

Shape finale : (15287, 37)
  17 colonnes tickets sources
  7 colonnes agents
  4 colonnes catégories
  5 features temporelles
  4 variables cibles/flags

Aperçu des 3 premières lignes :


,ticket_id,created_at,category_id,agent_id,pays,canal,priorite,statut,sla_heures,first_response_heures,resolution_heures,sla_breach,nb_contacts,reopened,csat,in_backlog,ratio_sla,nom,tier,tier_num,bureau,csat_moyen,taux_resolution_pct,tickets_actifs,nom_cat,domaine,priorite_defaut,sla_strict,heure_creation,jour_semaine,mois,est_weekend,heure_hors_bureau,ticket_at_risk,is_high_priority,is_long_wait,is_reopened
0,TKT000001,2022-01-01 08:06:00,CAT006,AGT001,Ghana,Email,3,Backlog,360,29.10,431.40,1,1,0,NaN,1,1.20,Aminata Diallo,Tier 1,1,Abidjan,4.20,88,42,Qualite produit,Product,3,False,8,5,1,1,0,1,0,1,0
1,TKT000002,2022-01-01 13:45:00,CAT001,AGT004,CI,Téléphone,4,Résolu,240,56.20,401.60,1,1,0,3.80,0,1.67,Moussa Kone,Tier 1,1,Abidjan,3.50,65,55,Facturation,Billing,4,True,13,5,1,1,0,1,0,1,0
2,TKT000003,2022-01-01 08:48:00,CAT008,AGT003,Maroc,Chat,2,Résolu,120,5.00,72.60,0,3,0,4.40,0,0.60,Fatou Sow,Tier 2,2,Dakar,4.50,94,28,Probleme paiement,Billing,2,True,8,5,1,1,0,0,1,1,0
3,TKT000004,2022-01-01 19:05:00,CAT007,AGT012,France,Email,5,En cours,480,139.50,803.00,1,3,0,NaN,0,1.67,Samuel Acheampong,Tier 2,2,Accra,4.40,92,26,Demande info,General,5,False,19,5,1,1,1,1,0,1,0
4,TKT000005,2022-01-01 22:54:00,CAT007,AGT001,Maroc,Téléphone,5,Escaladé,480,68.40,485.90,1,3,1,NaN,0,1.01,Aminata Diallo,Tier 1,1,Abidjan,4.20,88,42,Demande info,General,5,False,22,5,1,1,1,1,0,1,1


Regardons les 5 premières lignes.

`TKT000001` — ticket en Backlog, `sla_breach = 1`, `ratio_sla = 1.198`, `ticket_at_risk = 1`.

Agent Aminata Diallo, Tier 1, bureau Abidjan. Catégorie Qualité produit, domaine Product.

Créé un samedi (`est_weekend = 1`), à 8h du matin.

`TKT000003` — ticket Résolu, `sla_breach = 0`, `ratio_sla = 0.605`, `ticket_at_risk = 0`.

Agent Fatou Sow, Tier 2, bureau Dakar. CSAT de 4.4. Résolution en 72.6h sur un SLA de 120h.

C'est un bon ticket — résolu rapidement, client satisfait.

Le contraste entre ces deux tickets montre déjà ce que le modèle ML va devoir apprendre à distinguer.m

---

# 6. Export du dataset final

> 🎓 **Deux destinations, deux usages différents.**
>
> **Export CSV** → pour la portabilité et le partage.
> Le fichier `support_clean_analytics.csv` peut être ouvert par n'importe quel outil :
> Excel, Python, R, DuckDB, Power BI. C'est le format universel.
>
> **Export SQL Server** → pour la production et la collaboration.
> En entreprise, les données ne restent pas dans des CSV. Elles vivent dans des bases SQL
> accessibles à toute l'équipe data. L'export vers SQL Server permet au Notebook 3
> d'interroger les données avec des requêtes SQL analytiques avancées — window functions,
> CTE, agrégations complexes.
>
> **Si vous n'avez pas SQL Server installé :** sautez la section SQL et utilisez uniquement
> le CSV. DuckDB dans le Notebook 3 peut lire directement le CSV avec `read_csv_auto`.
> Les requêtes SQL sont exactement les mêmes.

### Export CSV

In [13]:
# Export CSV — toujours inclure index=False pour éviter une colonne d'index non désirée
final_df.to_csv(f'{SAVE_PATH}support_clean_analytics.csv', index=False)

print(f"✅ Fichier exporté : support_clean_analytics.csv")
print(f"   {len(final_df):,} lignes × {len(final_df.columns)} colonnes")
print(f"   Taux ticket_at_risk : {final_df['ticket_at_risk'].mean()*100:.1f}%")

# Vérification rapide : relire le CSV pour confirmer la bonne écriture
check = pd.read_csv(f'{SAVE_PATH}support_clean_analytics.csv', nrows=3)
print(f"\n✅ Vérification relecture : {len(check)} lignes lues correctement")
print(f"   Colonnes : {list(check.columns[:5])}...")

✅ Fichier exporté : support_clean_analytics.csv
   15,287 lignes × 37 colonnes
   Taux ticket_at_risk : 54.0%

✅ Vérification relecture : 3 lignes lues correctement
   Colonnes : ['ticket_id', 'created_at', 'category_id', 'agent_id', 'pays']...


---

# 7. Synthèse

> 🎓 **Quatre principes à retenir de ce notebook.**
>
> **Principe 1 — Toujours justifier chaque décision de nettoyage.**
> Médiane plutôt que moyenne pour les délais → because distribution asymétrique.
> Left join plutôt qu'inner join → pour garder tous les tickets traçables.
> Union `|` pour la variable cible → pour capturer toutes les formes de risque.
> Ces justifications doivent être dans le code ou dans des commentaires.
>
> **Principe 2 — Distinguer anomalie technique et anomalie métier.**
> Un délai négatif est une anomalie **technique** (bug de calcul) → on impute.
> Un CSAT null sur un ticket en cours est une anomalie **métier** (valeur non applicable) → on conserve.
> La confusion entre les deux mène à des imputations incorrectes.
>
> **Principe 3 — La variable cible ML se définit AVANT le feature engineering.**
> Si on crée des features après la variable cible et qu'une feature encode indirectement
> la cible (data leakage), le modèle va apprendre à "tricher" et sera inutilisable en production.
> L'ordre est impératif : nettoyage → cible → features.
>
> **Principe 4 — Penser réutilisabilité du dataset final.**
> Un dataset bien structuré sert pour l'analyse SQL, le dashboard Power BI ET le ML.
> Regrouper les colonnes par thème, nommer explicitement les variables créées, et exporter
> à la fois en CSV et en SQL : c'est l'investissement qui fait gagner du temps dans tous
> les notebooks suivants.

In [14]:
# Rapport final du dataset produit
print("=" * 60)
print("  RAPPORT FINAL — DATASET support_clean_analytics")
print("=" * 60)
print(f"  Lignes         : {len(final_df):,}")
print(f"  Colonnes       : {len(final_df.columns)}")
print(f"  Période        : {final_df['created_at'].min().date()} → {final_df['created_at'].max().date()}")
print(f"  Pays couverts  : {final_df['pays'].nunique()}")
print(f"  Agents         : {final_df['agent_id'].nunique()}")
print(f"  Catégories     : {final_df['category_id'].nunique()}")
print()
print("  VARIABLE CIBLE :")
print(f"    ticket_at_risk = 1 : {final_df['ticket_at_risk'].sum():>5,} ({final_df['ticket_at_risk'].mean()*100:.1f}%)")
print(f"    ticket_at_risk = 0 : {(final_df['ticket_at_risk']==0).sum():>5,} ({(1-final_df['ticket_at_risk'].mean())*100:.1f}%)")
print()
print("  NULLS RÉSIDUELS SUR COLONNES CRITIQUES :")
for col in ['sla_breach', 'ratio_sla', 'resolution_heures', 'first_response_heures']:
    n = final_df[col].isnull().sum()
    print(f"    {col:<30} : {n} {'✅' if n == 0 else '⚠️'}")

print()
print("  PRÊT POUR :")
print("    → Notebook 3 : SQL Analytics (DuckDB ou SQL Server)")
print("    → Notebook 4 : Power BI Dashboard")
print("    → Notebook 5 : ML — Détection de tickets à risque")

  RAPPORT FINAL — DATASET support_clean_analytics
  Lignes         : 15,287
  Colonnes       : 37
  Période        : 2022-01-01 → 2024-06-30
  Pays couverts  : 8
  Agents         : 12
  Catégories     : 10

  VARIABLE CIBLE :
    ticket_at_risk = 1 : 8,250 (54.0%)
    ticket_at_risk = 0 : 7,037 (46.0%)

  NULLS RÉSIDUELS SUR COLONNES CRITIQUES :
    sla_breach                     : 0 ✅
    ratio_sla                      : 0 ✅
    resolution_heures              : 0 ✅
    first_response_heures          : 0 ✅

  PRÊT POUR :
    → Notebook 3 : SQL Analytics (DuckDB ou SQL Server)
    → Notebook 4 : Power BI Dashboard
    → Notebook 5 : ML — Détection de tickets à risque


---

## Récapitulatif des opérations réalisées

| Étape | Opération | Justification |
|---|---|---|
| Audit | Rapport synthétique qualité | Vérifier l'état des données avant traitement |
| Audit | Contrôles cohérence métier | Détecter incohérences sla_breach/ratio_sla |
| Nettoyage | Déduplication clé primaire | Éviter le double comptage des tickets |
| Nettoyage | Imputation médiane (délais < 0) | Médiane robuste aux outliers de la distribution |
| Nettoyage | Normalisation statut (.str.strip) | Supprimer les espaces parasites de CRM |
| Fusion | Left join agents + catégories | Conserver tous les tickets traçables |
| Feature eng. | Décomposition temporelle (5 features) | Rendre les patterns horaires/hebdo explicites |
| Feature eng. | `ticket_at_risk` (variable cible) | Définir ce que le ML doit prédire |
| Feature eng. | 3 flags business (is_*) | Filtres rapides pour SQL et Power BI |
| Export | CSV | Portabilité (CSV)  |

---

**➡️ Prochaine étape → Notebook 3 : SQL Analytics & KPIs de performance**

---

*DataProjectLab — apprendre la data sur des cas concrets, structurés et orientés métier.*